# LAB 9: 비정형 데이터와 CORTEX AI FUNCTION

👉 이 수업에서는 제공된 PDF 문서를 검토하고, Snowflake Cortex AI Function을 사용해 해당 문서를 파싱한 뒤 정보를 테이블에 저장합니다. 또 다른 Cortex AI Function을 활용해 테이블에 저장된 데이터의 각 로우에 레이블을 지정(또는 분류)합니다. 이후 분석 쿼리를 작성하고, Streamlit을 사용해 카테고리별 정보 분포를 시각화합니다. 마지막으로, 다른 세 가지 Cortex AI Function의 주요 기능을 간단히 살펴봅니다. 

시작하기에 앞서, 이 실습 전반에서 사용할 **컨텍스트 정보**를 가져오겠습니다. 

- **Start** 버튼을 클릭하여 이 노트북을 활성화하세요.

- 다음 Python 셀을 실행하세요.

#### :warning: 이 노트북에 대해 새 세션이 시작될 때마다, 후속 셀에서 사용할 '변수'를 구성하기 위해 아래 셀을 다시 실행해야 합니다. :warning:

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
session.use_database(f'{user}_GARDEN_PLANTS')
session.use_schema('VEGGIES')
print('현재 CONTEXT 정보:')
print('---------------------------------')
print(session)
print('현재 USER는 ' + user)

## Snowflake에서 비정형 데이터 다루기 📓

이 과정의 이전 섹션에서는 정형 데이터(로우와 컬럼)와 반정형 데이터(JSON 형식)를 모두 살펴보았습니다. Snowflake는 [**비정형 데이터**](https://docs.snowflake.com/ko/user-guide/unstructured-intro) 작업을 위한 기능과 함수도 제공하며, 분석가들은 현재 전 세계 데이터의 약 80%가 비정형 데이터라고 추정합니다.

그렇다면 비정형 데이터란 정확히 무엇일까요?

**비정형 데이터**는 사전 정의된 데이터 모델이나 스키마에 맞지 않는 정보입니다. 비정형 데이터에는 양식 응답이나 소셜 미디어 대화와 같이 텍스트가 많으며, 이미지, 비디오, 오디오도 포함됩니다. VCF(유전체학), KDF(반도체), HDF5(항공우주)와 같은 산업별 파일 유형도 이 범주에 속합니다.

Snowflake AI Data Cloud는 **비정형 데이터** 파일에 액세스하고, 공유하고, 처리하는 데 도움이 됩니다.

## 비정형 데이터 시나리오 📓

잠시 상상해 보세요. 여러분이 정성껏 돌보는 식물들에 대해 흥미로운 사실과 세부 정보들을 오랫동안 수집해 왔고, 이 정보 '조각(snippets)'들을 PDF 파일로 노트북에 저장해 두었다고 말이죠. 이러한 데이터를 Snowflake로 가져와 기존에 구축해 둔 정보와 함께 활용할 수 있다면 얼마나 좋을까요?

바로 이 시나리오를 이번 실습에서 다뤄보겠습니다. 다음을 수행하게 됩니다.
1. Cortex AI Function을 사용하여 스테이징된 PDF 파일의 정보를 파싱합니다.
1. 데이터를 추출하여 Snowflake의 새 테이블에 수집합니다.
1. Snowflake Cortex AI Function을 활용해 데이터를 분석하고, 새로운 결과를 생성해 봅니다. :robot_face:

### 제공된 PDF 파일의 목록을 생성합니다. 🥋

Education Services 팀에서 식물에 대한 흥미로운 사실과 세부 정보가 담긴 약 130개의 PDF 파일 모음을 **common_db.resources**의 **course_files** 스테이지에 업로드했습니다.

아래의 SQL 코드 발췌본을 **수정**하고 **셀을 실행**하여, 지정된 스테이지에 있는 PDF 파일들을 `LIST`하세요.
- 해시 기호 `(#)`를 적절한 명령과 구문으로 바꿔 올바른 SQL 구문을 완성하세요.
- SQL 셀을 실행하세요.

In [ ]:
%%sql -r dataframe_1
#### @common_db.resources.course_files/garden_kb;

## Directory Table 📓

비정형 데이터를 다룰 때 **Directory Table**은 매우 유용하며, 여러 면에서 `LIST` 명령을 사용하는 것보다 훨씬 뛰어납니다.

[Directory Table](https://docs.snowflake.com/ko/user-guide/data-load-dirtables)은 스테이지 위에 계층화된 암시적 오브젝트(별도의 데이터베이스 오브젝트가 아님)로, 스테이지 내 데이터 파일에 대한 파일 수준의 메타데이터를 저장한다는 점에서 개념적으로 External Table과 유사합니다. Internal Stage 및 External Stage를 모두 지원합니다. 가장 큰 장점은 이러한 오브젝트를 통해 디렉터리의 내용을 마치 일반 테이블처럼 쿼리할 수 있다는 점입니다. Directory Table을 만드는 과정은 매우 간단해서, 스테이지를 생성할 때 옵션을 활성화하거나 생성 후 스테이지를 수정하기만 하면 됩니다. 

- 사실 여러분은 Lab 8에서 이미 Directory Table을 생성했습니다. Snowsight 마법사를 사용해 스테이지 오브젝트를 만들 때 기본적으로 포함되는 옵션이라 미처 깨닫지 못했을 수도 있습니다.


테이블 함수를 사용해 이 Directory Table의 출력값에 액세스할 수 있으며, 파일 이름과 위치 정보를 다양한 Snowflake 함수에 간편하게 전달하여 처리할 수 있습니다.

### Directory Table을 쿼리하세요. 🥋

다음 쿼리의 구조를 살펴보세요.
- 이것은 일반적인 `SELECT` 구문입니다.
- 테이블 함수에 액세스하기 위해 **DIRECTORY** 키워드를 사용하는 것을 확인하세요.
- 이번 경우에는 모든 컬럼을 반환하도록 `*`(전체 선택)를 사용할 수 있습니다.
- 또한 결과에 필터를 적용할 수도 있습니다. 이 스테이지의 'garden_kb' 하위 디렉터리에 있는 파일만 반환하도록 설정한 것처럼 말이죠.

이 쿼리를 실행한 뒤, 해당 스테이지 위치에 포함된 파일들에 대해 어떤 유형의 정보가 반환되는지 출력 결과를 확인해 보세요.

In [ ]:
%%sql -r dataframe_2
SELECT *    
FROM DIRECTORY('@common_db.resources.course_files')
WHERE CONTAINS(relative_path, 'garden_kb/');

## 제공된 PDF 파일을 검토하세요. 🥋

### 어떤 원예(gardening) 관련 정보가 수집되었나요?

앞서 언급했듯이 Education Services 팀에서 식물에 대한 흥미로운 사실과 세부 정보가 담긴 약 130개의 PDF 파일 모음을 업로드했습니다.

- PDF 파일 모음의 파일 이름은 **snippet_1.pdf**부터 **snippet_129.pdf**까지입니다. 

이러한 '사실 정보(factoid)' 파일에 어떤 내용이 담겨 있는지 궁금하실 수 있습니다.

다음은 식물 및 원예 지식 PDF 파일 모음에서 발췌한 예시입니다. 아래 텍스트는 모음의 첫 번째 파일에서 가져온 것으로 **artichokes**에 관한 내용입니다.

**artichokes**를 재배하시나요? 유용한 팁이 여기 있습니다.

![snippet 샘플(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_artichokes_1.png)

### 식물 '사실 정보(factoid)' PDF 파일 중 하나를 다운로드해서 살펴보세요. 📓

위에서 식물 및 원예 지식 모음의 PDF 파일 내용 예시를 살펴보았습니다. 이제 다른 PDF 파일을 다운로드하여 열어보고 살펴보겠습니다.

- 다음 예시에서는 **snippet_9.pdf** 파일을 살펴보겠습니다.

- 이 예시의 SQL 코드(Python 안에 포함됨)는 스테이지에 저장된 파일에 대한 링크를 생성하는 Snowflake 함수를 사용합니다. 이 링크는 Snowflake 외부에서도 액세스 가능합니다. 이는 **비정형** 데이터를 다룰 때 매우 유용합니다. 자세한 내용은 Snowflake 설명서에서 [GET_PRESIGNED_URL](https://docs.snowflake.com/ko/sql-reference/functions/get_presigned_url) 함수에 대해 참조하세요.

### 다음 Python 코드 셀을 실행하고, 생성된 링크를 사용하여 파일을 다운로드하세요. 🥋

생성된 링크 위에서 마우스 **오른쪽 버튼을 클릭**한 뒤 새 탭 또는 브라우저 창에서 **Open**을 선택하세요(링크를 단순히 클릭하면 작동하지 않음).

브라우저에서 자동으로 열리지 않으면, 로컬 기기의 PDF 뷰어 애플리케이션으로 파일을 **여세요**.

- 이제 이 PDF 파일 안에 여러분이 좋아하는 채소인 아스파라거스에 대해 어떤 흥미로운 정보들이 들어 있는지 살펴봅시다. :leafy_green:

- 첫 번째 예시를 완료한 후에는, 1번부터 129번까지 번호가 매겨진 다른 파일들도 자유롭게 살펴보세요.

In [ ]:
snowpark_df = session.sql("SELECT GET_PRESIGNED_URL(@common_db.resources.course_files, 'garden_kb/snippet_9.pdf')")
collected_data = snowpark_df.collect()
st.write('다음 링크를 새 브라우저 탭 또는 창에서 열고 검토하세요.')
st.write(collected_data[0][0])
st.write('Asparagus. Who knew!!!')

## Snowflake Cortex AI 소개 📓

Snowflake AI Data Cloud에는 Mistral, Reka, Meta, Google과 같은 기업의 연구진이 학습시킨 업계 최고의 대규모 언어 모델(LLM)에 즉시 액세스할 수 있게 해주는 다양한 기능과 함수가 포함되어 있습니다. 또한 Snowflake가 개발한 오픈 엔터프라이즈급 모델인 Snowflake Arctic도 포함되어 있습니다.

이러한 LLM은 Snowflake에서 완벽하게 호스팅 및 관리되므로 별도의 **설정이 필요하지 않습니다**. 데이터는 Snowflake 내에 유지되며 기대하는 성능, 확장성, 거버넌스를 제공합니다.

💡 다음 기능들의 [출시 상태 및 가용성](https://docs.snowflake.com/ko/guides-overview-ai-features)은 Snowflake 설명서를 참조하세요.

![Snowflake 생성형 AI(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_gen_ai_1_v2.png)

### Snowflake Cortex AI Function:

**Cortex AI Function**은 Snowflake AI 옵션의 하위 집합 중 하나입니다. 이 함수들은 SQL 함수 형태로 제공되며, Python에서도 사용 가능하여 강력한 기능들을 매우 쉽게 활용할 수 있습니다.

Cortex AI Function은 다음과 같은 카테고리로 분류할 수 있습니다.

- 작업 특화 함수

- Helper 함수

- `COMPLETE` 함수

👉 이 실습의 나머지 부분에서는 이러한 함수 중 일부를 사용하여 식물 및 원예 데이터를 수집하고 처리할 것입니다.

---

### 제1세대 Snowflake Cortex 함수

제1세대 Cortex 함수는 **Cortex LLM Functions라고** 불렸습니다. 

이러한 함수들은 여전히 플랫폼에서 사용할 수 있지만, 최신 기능을 이용하려면 **업데이트된**  `AI_`  **버전을** 사용하세요.

- [PARSE_DOCUMENT](https://docs.snowflake.com/ko/sql-reference/functions/parse_document-snowflake-cortex) **=>** AI_PARSE_DOCUMENT
- [CLASSIFY_TEXT](https://docs.snowflake.com/ko/sql-reference/functions/classify_text-snowflake-cortex) **=>** AI_CLASSIFY
- [TRANSLATE](https://docs.snowflake.com/ko/sql-reference/functions/translate-snowflake-cortex) **=>** AI_TRANSLATE
- [COMPLETE](https://docs.snowflake.com/ko/sql-reference/functions/complete-snowflake-cortex) **=>** AI_COMPLETE

## `AI_PARSE_DOCUMENT()`를 활용하여 텍스트 추출하기 📓

Cortex [AI_PARSE_DOCUMENT()](https://docs.snowflake.com/ko/sql-reference/functions/ai_parse_document)는 Internal Stage 또는 External Stage에 저장된 문서에서 텍스트 또는 레이아웃을 추출할 수 있는 Cortex AI 작업 특화 함수입니다. 

이 함수는 SQL 함수입니다. Snowflake에서 완벽하게 호스팅 및 관리되므로 별도의 설정이 필요하지 않습니다. PDF 문서가 저장된 스테이지를 `AI_PARSE_DOCUMENT` 함수로 지정하기만 하면, 텍스트 또는 레이아웃 데이터를 손쉽게 추출할 수 있습니다. 간단히 말하면, 필요한 것은 다음과 같습니다.

- 읽어올 스테이지의 이름

- 해당 스테이지 내에서 텍스트를 추출하려는 PDF 문서(현재는 PDF 파일만 지원됨)

- 문서 레이아웃도 읽을 수 있지만, 우리는 텍스트 추출에 적합한 **OCR** 모드를 선택하여 작업할 것입니다.

💡 **팁**: 보다 정교한 문서 추출 사용 사례를 위해서는 이 함수의 **LAYOUT** 모드를 살펴보거나, Snowflake의 [Cortex AI Document](https://docs.snowflake.com/ko/user-guide/snowflake-cortex/ai-documents) 서비스를 검토해 보는 것이 좋습니다.

### 텍스트 추출 예시 🥋

다음 SQL 구문을 검토하세요.

- 이 구문은 `AI_PARSE_DOCUMENT()` 함수에 대해 전체 경로 참조를 사용하고 있습니다.

- PDF 문서 스테이지와 그 안의 단일 파일 하나를 참조하도록 되어 있습니다(앞서 열어본 '아스파라거스' 예시).

이토록 간단합니다. 이 '빌트인' SQL 함수를 호출하기만 하면 Snowflake가 보이지 않는 곳에서 파일을 열고 내용을 읽어 결과를 반환합니다.

이제 다음 코드를 실행해 보세요.

In [ ]:
%%sql -r dataframe_3
SELECT SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT (
        TO_FILE('@common_db.resources.course_files','garden_kb/snippet_9.pdf'),
        {'mode': 'OCR'}
    ) AS output;

### `AI_PARSE_DOCUMENT()` 출력을 검토하세요. 🥋

이 함수의 출력은 반정형 데이터 형식으로 제공됩니다.

![AI_PARSE_DOCUMENT() 출력(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_parse_document_1.png)

**content** 필드 및 **metadata** 필드가 있습니다. 

💡 **팁**: **LAB 8**에서 배웠듯이 `:` 연산자를 사용하여 이러한 반정형 데이터 구조에 포함된 중첩 필드를 자세히 살펴볼 수 있다는 것을 기억하세요.

## 텍스트 추출 워크플로우 구축 📓

제공된 식물 및 원예 관련 PDF 문서에서 텍스트를 추출하는 방법을 알았으므로, 이제 추출한 텍스트를 Snowflake로 가져와 활용하려고 합니다.

👉 수집한 모든 식물 및 원예 정보를 기반으로 **지식 베이스**를 구축하고자 하며, 이 데이터는 다양한 용도로 활용될 수 있습니다.

워크플로우는 다음과 같습니다.

- 지식 베이스 정보는 PDF 파일의 **비정형** 데이터로 시작합니다.

- Cortex `AI_PARSE_DOCUMENT()` 함수를 실행하여 PDF 파일에서 텍스트를 추출하고 **반정형** 데이터 형식으로 변환합니다.

- Snowflake 구문을 사용하여 Cortex 함수의 출력을 구문 분석하여 **정형** 데이터 콘텐츠를 반환합니다.

### 지식 베이스 데이터를 저장할 테이블을 생성하세요. 🥋 

먼저 PDF 문서에서 추출한 참조 정보를 저장할 테이블을 생성해야 합니다.

**(animal)_GARDEN_PLANTS.VEGGIES** 스키마에서 다음 코드를 실행하여 테이블을 생성하세요. PDF 파일 콘텐츠에서 참조하는 식물 이름을 위한 컬럼을 추가할 예정입니다. 

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE vegetable_knowledge_base (
    source_document STRING, -- 문서 이름
    insight STRING,         -- 파일에 포함된 "factoid"
    plant_name STRING       -- 그 "factoid"가 참조하는 식물의 이름
);

### 데이터를 추출한 다음, 새 테이블에 `INSERT` 하세요. 🥋

이제 이 실습에서 배운 몇 가지 기능과 함수를 함께 사용해 보겠습니다.

다음 `INSERT` 구문은 **Directory Table**과 `AI_PARSE_DOCUMENT()` 함수를 활용합니다. 

- **Directory Table** 목록은 **common_db.resources.course_files** 스테이지의 **garden_kb** 하위 디렉터리에 있는 각 PDF 파일의 이름을 반환합니다.

- 각 PDF 문서의 위치가 `AI_PARSE_DOCUMENT()` 함수에 전달되어 텍스트가 추출됩니다.

- 추출된 텍스트는 반정형 형식으로 반환되며, 그중 **content** 엘레멘트는 따로 분리되어 `:content::STRING as extract` 구문을 사용해 `STRING` 타입으로 형 변환됩니다.

아래 코드를 **실행**하여 각 PDF 파일의 파일 이름과 추출된 텍스트를 새 테이블에 기록합니다.

In [ ]:
%%sql -r dataframe_5
INSERT INTO vegetable_knowledge_base (source_document, insight)
    SELECT 
        split_part(relative_path,'/',-1) as file_name, 
        SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT (
            TO_FILE('@common_db.resources.course_files',
            relative_path),
            {'mode': 'OCR'}
        ):content::STRING as extract    
    from directory('@common_db.resources.course_files')
    where contains(relative_path, 'garden_kb/')
;

### 작업 결과를 확인하세요. 🎯

새 테이블에 **129**개의 로우가 삽입되어야 합니다. 이제 테이블에 있는 데이터의 '형태'를 살펴보세요.

- 다음 쿼리 조각을 **재작성**하여 새 지식 베이스 테이블에서 **모든** 로우를 반환하도록 합니다.

In [ ]:
%%sql -r dataframe_6
SELECT #
FROM #########_#########_####;

### 문제가 발생했습니다.

제공된 PDF 파일에서 데이터를 추출하여 테이블에 기록하는 데는 성공했지만, 문제가 있습니다.

**INSIGHT** 컬럼의 콘텐츠를 검토하지 않고는 각 로우의 정보가 어떤 식물과 관련된 것인지 알 수 없습니다. **PLANT_NAME** 컬럼이 모두 비어 있습니다. 이는 지식 베이스를 구현할 때 심각한 문제입니다.

각 로우를 _읽고_ **PLANT_NAME** 컬럼을 수동으로 업데이트할 수도 있지만, 이 테이블에 129개의 로우가 있는 경우 시간이 매우 오래 걸리고, 데이터 세트가 훨씬 더 커지면 이러한 방식은 실행 불가능합니다.

![지식 베이스 테이블 1(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_kb_table_query_1.png)

## 추출한 텍스트에 `AI_CLASSIFY()` 활용하기 📓

다행히 Snowflake Cortex에는 [AI_CLASSIFY()](https://docs.snowflake.com/ko/sql-reference/functions/ai_classify)라는 AI Function이 포함되어 있습니다. 이름에서 알 수 있듯이 이 함수는 사용자가 제공하는 자유 형식 텍스트 데이터를 분류합니다.

- SQL에서 간단하게 호출할 수 있으며, 다른 Cortex AI Function과 마찬가지로 별도의 설정이 필요하지 않습니다. Snowflake 제품에 포함되어 있습니다.

- 이 함수는 JSON 오브젝트를 포함하는 문자열을 반환합니다. JSON 오브젝트에는 입력 프롬프트가 분류된 카테고리가 포함되어 있습니다. 유효하지 않은 인수가 제공되면 오류가 반환됩니다.

![AI_CLASSIFY 사용법 (이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_classify_text_1.png)

### 분류 예시 🥋

다음은 우리의 사용 사례와 관련된 예시입니다. 입력 문자열 **apple**을 Cortex AI Function에 전달하여 이를 검토하고, **fruit**, **veggie**, **flower** 중 어떤 분류에 해당하는지 올바르게 분류하길 기대합니다.

이제 다음 코드를 실행해 보세요.

- 정확했나요?

- 이번에는 입력값으로 **tomato**를 사용해 보세요. :grinning:

In [ ]:
%%sql -r dataframe_7
SELECT AI_CLASSIFY('apple', ['fruit', 'veggies', 'flowers']);

### 확장된 분류 사용 사례 📓

아주 훌륭합니다. 이는 테이블에 있는 129개의 로우를 수동으로 업데이트할 필요가 없고, Snowflake가 그 복잡한 작업을 대신 처리해 줄 수 있다는 의미입니다.

- `UPDATE` 구문을 작성해야 합니다. 

- 테이블의 각 로우에 대해 **INSIGHT** 컬럼을 입력값으로 전달합니다.

- 이전 실습에서 생성하고 데이터를 로드했던 **VEGETABLE_DETAILS** 테이블에서, 우리가 참조를 저장해 둔 모든 알려진 식물의 이름 목록을 구성할 수 있습니다.

- 반정형 데이터 출력 값을 더 깊이 탐색하여 ​​레이블 값을 추출할 수 있습니다.

### 분류를 실행하세요. 🥋

다음 SQL 구문을 검토하세요.

- 이 구문은 `AI_CLASSIFY()` 함수에 대해 전체 경로 참조를 사용하고 있습니다.

 - **4번 라인**의 구문이 다소 낯설게 느껴질 수 있지만, 이 코드는 단순히 **vegetable_details** 테이블에서 식물 이름을 모아 Cortex AI Function에 전달하기 위한 `ARRAY`를 생성하는 것입니다.
    - 이렇게 하면 `Artichoke...Zucchini`를 일일이 수동으로 입력하는 수고를 덜 수 있습니다.

이제 다음 코드를 실행해 보세요.

In [ ]:
%%sql -r dataframe_8
UPDATE vegetable_knowledge_base
SET plant_name = AI_CLASSIFY(
    insight, 
    (SELECT ARRAY_AGG(plant_name) WITHIN GROUP (ORDER BY plant_name ASC) FROM vegetable_details) -- ASSEMBLE CATEGORIES
):labels[0]::STRING
WHERE insight IS NOT NULL 
AND insight <> '';

### 작업 결과를 확인하세요. 🥋

지식 베이스 테이블에서 **129**개의 로우가 **업데이트**되었을 것입니다. 테이블의 데이터를 확인해 보세요.

- 다음 쿼리를 실행하여 지식 베이스 테이블의 모든 로우를 반환하세요.

- **PLANT_NAME** 컬럼이 **모든** 로우에 채워졌는지 확인하세요.

In [ ]:
%%sql -r dataframe_9
SELECT *
FROM vegetable_knowledge_base;

## 지식 베이스 데이터 분석 📓

제공된 PDF 파일에서 로드한 지식 베이스 테이블은 각 로우가 식물 이름에 따라 분류되어 있습니다. 이제 이 데이터의 분포를 이해하는 것이 유용할 것입니다.

- 식물별로 우리가 가지고 있는 '사실 정보(factoid)'는 몇 개일까요? 

- 어떤 식물에 대해서는 다른 식물보다 더 많은 정보가 있을까요?

- 또 일부 식물에 대해서는 데이터가 누락되어 있을까요?

이 모든 내용은 지식 베이스를 확장하기 위한 우선순위를 정하는 데 도움이 됩니다.

### 분석 쿼리를 실행합니다. 🥋

다음 쿼리는 Snowflake에서 지원하는 [JOIN](https://docs.snowflake.com/ko/sql-reference/constructs/join) 연산 중 하나의 예시입니다.

여기서는 **vegetable_details** 테이블에서 모든 식물 이름의 전체 목록을 가져오고자 합니다. 각 식물마다 **vegetable_knowledge_base** 테이블 내의 로우 수를 계산해 표시합니다. 이때 `LEFT OUTER JOIN`을 사용하면, 해당 식물에 대한 로우가 **vegetable_knowledge_base**에 없더라도 해당 항목에 0 값을 할당할 수 있으며, 결과에서 완전히 제외되는 것을 방지할 수 있습니다.

다음 쿼리를 실행하고 결과를 검토하여 식물별 기사 수를 확인해 보세요.

💡 **팁**:  나중에 출력을 쉽게 참조할 수 있도록 이 셀의 이름을 **knowledge_base_analytical_query**로 지정합니다.

In [ ]:
%%sql -r dataframe_10
SELECT a.plant_name, 
       nvl(count(b.*),0) AS kb_article_count
FROM vegetable_details a
LEFT OUTER JOIN vegetable_knowledge_base b
ON a.plant_name = b.plant_name
GROUP BY a.plant_name
ORDER BY 1;

## Streamlit in Snowflake 📓

[Streamlit](https://streamlit.io/)은 머신러닝 및 데이터 사이언스를 위한 맞춤형 웹 앱을 쉽게 만들고 공유할 수 있도록 지원하는 오픈 소스 Python 라이브러리입니다.

[Streamlit in Snowflake](https://docs.snowflake.com/ko/developer-guide/streamlit/about-streamlit)은 이 기술을 Snowflake AI 데이터 클라우드 상에 구현한 것입니다. 

- Snowflake는 Streamlit 앱의 기본 컴퓨팅 및 스토리지를 관리합니다.

- Streamlit 앱은 Snowflake 오브젝트이며, 역할 기반 액세스 제어(RBAC)를 사용하여 Streamlit 앱에 대한 액세스를 관리합니다.

- Streamlit 앱은 Snowflake 웨어하우스에서 실행되며 Internal Stage를 사용하여 파일과 데이터를 저장합니다.

### Streamlit으로 데이터를 시각화하세요. 🥋

Streamlit의 가장 큰 장점은 데이터를 손쉽게 시각화하고 상호작용할 수 있다는 점입니다. 이 과정의 각 실습 마지막에 있는 **퀴즈** 섹션에서도 이미 이를 사용해 보았습니다. 단 몇 줄의 Python 코드만으로, 데이터를 생동감 있게 표현하는 깔끔하고 매력적인 차트와 그래프를 만들 수 있습니다.

다음 Python 셀을 실행하면, 방금 수행한 분석 쿼리 결과를 시각화하는 Streamlit 막대 차트가 생성됩니다. 이 그래프를 만드는 데 필요한 코드가 얼마나 간단한지 확인해 보세요.

- 어떤 식물에 지식 베이스 기사가 없나요?

- 또, 가장 많은 정보를 가진 식물은 어떤 것인가요?

💡 **팁**: 이 과정 초반에 배웠듯이 Snowflake Notebook에서는 이전 셀의 결과를 나중 셀에서 참조할 수 있습니다. 다음 코드는 이 기능을 활용하여, 방금 실행한 분석 SQL 쿼리 셀(**knowledge_base_analytical_query**)의 출력을 가져와 **pandas** DataFrame을 생성하고, 이를 Streamlit에 전달합니다.

In [ ]:
import streamlit as st
import pandas as pd

chart_data = knowledge_base_analytical_query.to_pandas() # 이전 SQL 셀의 출력을 사용합니다 !!!

st.header("Knowledge Base Articles Per Plant")
st.bar_chart(chart_data, x="PLANT_NAME", y="KB_ARTICLE_COUNT", color=['#33C4FF'])

## Snowflake의 AI 및 LLM Function에 대해 알아볼 내용이 더 많습니다. 📓

이 과정에서 다룰 수 있는 것보다 훨씬 더 많은 Snowflake의 AI(및 ML) 기능들이 존재합니다. 

하지만 그중에서도 특히 유용한 몇 가지 작업 특화형 Cortex AI Function들로는 `AI_TRANSLATE()`과 `SUMMARIZE()`가 있습니다. 이 함수들을 간단히 살펴보겠습니다.

- [AI_TRANSLATE()](https://docs.snowflake.com/ko/sql-reference/functions/ai_translate) - 주어진 입력 텍스트를 지원되는 한 언어에서 다른 언어로 번역

- [SUMMARIZE()](https://docs.snowflake.com/ko/sql-reference/functions/summarize-snowflake-cortex) - 주어진 영어 입력 텍스트 요약

이제 지식 베이스 데이터를 활용한 몇 가지 간단한 예시를 살펴보겠습니다.

### `AI_TRANSLATE()` 예시를 실행하세요. 🥋 

이전 예시에서 보았듯이 SQL을 사용하여 Cortex AI Function에 액세스하는 것은 간단합니다. `AI_TRANSLATE()`는 다른 SQL 함수와 마찬가지로 호출할 수 있으며, 다음과 같은 입력값을 제공합니다.

- 번역할 텍스트가 포함된 문자열

- 현재 텍스트의 언어 코드를 지정하는 문자열. 옵션에는 프랑스어, 독일어, 이탈리아어, 일본어, 한국어, 스페인어 등이 있습니다.

- 텍스트를 번역할 언어 코드를 지정하는 문자열

다음 예시 코드를 **실행**하여 이름이 `'C'`로 시작하는 모든 식물의 지식 베이스 정보를 **영어**에서 **한국어**로 번역한 다음 번역된 버전에서 다시 **영어**로 번역하세요.

In [ ]:
%%sql -r dataframe_11
SELECT plant_name, 
       insight AS original_english_text,
       AI_TRANSLATE(original_english_text, 'en', 'ko') AS korean_text,
       AI_TRANSLATE(korean_text, 'ko', 'en') AS english_text_from_korean,
FROM vegetable_knowledge_base
WHERE LEFT(plant_name,1) = 'C'; -- 모든 C로 시작하는 식물 이름

### `SUMMARIZE()` 예시를 실행하세요. 🥋 

이름에서 알 수 있듯이 `SUMMARIZE()` 함수는 주어진 영어 입력 텍스트에 대한 요약을 생성합니다. 파라미터는 하나만 받으며, 요약할 텍스트 문자열입니다. 

우리의 수집 목록 중 하나인 **Pumpkin**에 초점을 맞춘 다음 예시를 **실행**하세요.

- [LISTAGG](https://docs.snowflake.com/ko/sql-reference/functions/listagg)를 사용하여 이 식물에 대한 모든 '사실 정보(factoid)'(인사이트)를 하나의 텍스트 단위로 결합합니다.

- `SUMMARIZE()`는 이 단일 텍스트를 대상으로 실행되어 요약본을 생성합니다.

- 또한 Cortex AI Helper 함수인 [COUNT_TOKENS()](https://docs.snowflake.com/ko/sql-reference/functions/count_tokens-snowflake-cortex)를 사용하여 요약 **전**과 **후**의 텍스트의 상대적 길이를 확인합니다. 물론, 출력 결과는 직접 검토할 수도 있습니다.

💡 **팁**: 토큰은 Snowflake Cortex AI Function이 처리하는 가장 작은 텍스트 단위로, 대략 네 글자에 해당합니다. 원시 입력 또는 출력 텍스트와 토큰의 등가성은 모델에 따라 다를 수 있습니다.

이제 다음 코드를 실행하고 출력을 확인해 보세요.

In [ ]:
%%sql -r dataframe_12
SELECT listagg(insight) AS all_insights,
        SNOWFLAKE.CORTEX.count_tokens('summarize', all_insights) AS all_insights_tokens,
        SNOWFLAKE.CORTEX.SUMMARIZE(listagg(insight)) AS summary,
        SNOWFLAKE.CORTEX.count_tokens('summarize', summary) AS summary_tokens
FROM vegetable_knowledge_base
WHERE plant_name = 'Pumpkin';

## 그리고 마지막으로 `AI_COMPLETE()` 🥋

Cortex AI Function 중 가장 정교한 함수는 [AI_COMPLETE](https://docs.snowflake.com/ko/sql-reference/functions/ai_complete)입니다. 가장 단순한 형태에서 이 함수는 **프롬프트**(텍스트와 해당 텍스트로 수행할 작업에 대한 지침)를 입력으로 받아, 지원되는 언어 모델 중 하나를 사용하여 **응답**(completion)을 생성합니다.

작동 방식을 이해하기 위해, 다음 예시를 살펴보세요.

- 이번 예시에서는 **claude-sonnet-4-5** 모델을 사용합니다.

- 모델이 적절한 응답을 생성할 수 있도록, 우리는 모델에게 지침(예: “당신은 IT 전문가입니다” 또는 "You are an I.T expert")을 제공합니다.

- 질문을 제시합니다(예: “Snowflake가 무엇인지 설명해 주세요” 또는 "Explain what Snowflake is").

- 모델의 응답은 우리가 제공하는 추가 정보가 아니라, 모델이 학습한 데이터, 즉 모델 **본연의** 지식을 바탕으로 생성됩니다.

다음 코드를 실행하고 출력을 검토하세요.

In [ ]:
%%sql -r dataframe_13
SELECT AI_COMPLETE('claude-sonnet-4-5', 'You are an I.T expert. Explain what Snowflake is.') AS complete_response;

### `AI_COMPLETE()` 함수와 함께 지식 베이스 데이터를 사용하세요. 🥋

마지막 예시로, 지식 베이스에 있는 **모든** 정원 및 식물 관련 인사이트를 하나로 모아보겠습니다. 그리고 이 정보를 바탕으로 `AI_COMPLETE()` 함수가 답변을 생성할 수 있도록 질문을 던져보겠습니다. 

### `AI_COMPLETE()` 쿼리에 대한 설명은 다음과 같습니다.

다음 쿼리는 다소 복잡해 보일 수 있으므로 각 부분을 나누어 설명하겠습니다.

#### 섹션 1.

- 첫 번째 섹션에는 실습용 샘플 질문들이 포함되어 있습니다.

- SQL 셀을 실행할 때마다 하나씩 주석을 해제하여 새로운 질문을 던져보세요.

- 참고로, 이러한 쿼리는 실행하는 데 시간이 다소 걸릴 수 있으니 잠시만 기다려 주세요.

![쿼리 완성 섹션 1(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_complete_query_1.png)

#### 섹션 2.

- 두 번째 섹션에서는 프롬프트를 정의합니다. 프롬프트란 `COMPLETE()` 함수를 통해 모델에 전달되는 우리의 지침과 요청 내용을 의미합니다.

- 모델이 정보를 효율적으로 'consumption'할 수 있도록, 프롬프트 내의 서로 다른 텍스트 블록을 명확히 구분하기 위해 태그를 포함했다는 점에 유의해 주세요.

![쿼리 완성 섹션 2(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_complete_query_2.png)

#### 섹션 3.

- 세 번째 섹션은 Cortex 함수를 실제로 호출하는 부분입니다.

- 셀의 앞부분에서 정의한 변수를 사용하여 반복 작업을 더욱 유연하게 수행할 수 있습니다.

![쿼리 완성 섹션 3(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_complete_query_3.png)

In [ ]:
%%sql -r dataframe_14
-- 예제 질문
SET my_question  = 'which plant would take the shortest amount of time to cook';
--SET my_question  = 'which plants are best for beginner gardeners';


-- 프롬프트
SET prompt = 'You are a helpful gardening expert. Use only the supplied information <information> to answer the question posed <question>' || 
              $my_question || '</question> ' ||                 
             ' If you have no supplied information do not answer'; 
                
-- Cortex 함수 호출
SELECT AI_COMPLETE(
    'mixtral-8x7b', --32K context window
    $my_question || 
    '<information>' ||
    (SELECT LISTAGG(insight, ' ') FROM vegetable_knowledge_base) ||
    '</information>'
) AS cortex_output;

## 마지막 퀴즈 :mag_right:

## 지식 테스트 :mag_right:

아래의 대화형 퀴즈 문제를 통해 이해도를 확인해 보세요. 각 `RUN_THIS_QUIZ_QUESTION_` 셀에는 Snowflake 기능과 관련된 객관식 문제를 제시하는 Streamlit 위젯이 포함되어 있습니다.  

**지침:**  
1. 노트북 셀 위에 커서를 올려 추가 컨트롤을 표시하세요.
1. 각 퀴즈 셀 오른쪽의 ▶️ **Play 버튼**을 클릭하여 실행하세요.  
1. 제공된 옵션에서 답을 선택하세요.  
1. 다음으로 넘어가기 전에 피드백을 검토하세요.  

💡 **참고:** 궁금하시면 셀을 확장하여 코드를 볼 수 있지만, 필수는 아닙니다. 이 퀴즈들은 필수 사항이 아닙니다. 배운 내용을 복습하며 연습할 기회를 제공하기 위한 것입니다.  

:point_right: 질문 내용은 이번 과정에서 다룬 모든 주제를 포함할 수 있습니다.

In [ ]:
st.divider()
question = "과거에 작동했던 SELECT 문을 실행했는데, 이제 테이블이 존재하지 않는다는 오류 메시지가 나타나면 무엇을 확인해야 할까요?"
options = ["아래 선택을 고르세요...",
           "A) 클라우드 제공업체", 
           "B) 현재 지역", 
           "C) 컨텍스트 역할",
           "D) 컨텍스트 웨어하우스",
           "E) 화면 왼쪽 하단에 있는 이름 아래의 역할"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...":
        ''
    else:
        answer = 'ef3fa590e5b728d1bdaf658d3a18945c'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "이 중 가장 많은 컴퓨팅 파워를 가진 오브젝트는 무엇인가요?"
options = ["아래 선택을 고르세요...",
           "A) Snowflake 데이터베이스", 
           "B) Snowflake 스키마", 
           "C) Snowflake 시퀀스",
           "D) Snowflake 웨어하우스",
           "E) Snowflake 데이터 마트"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...":
        ''
    else:
        answer = 'fe742826f7ff02a3e33b22e228ea368e'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "저장 컨테이너 계층 구조에서 가장 낮은 단계의 오브젝트는 무엇인가요?"
options = ["아래 선택을 고르세요...",
           "A) Snowflake 데이터베이스", 
           "B) Snowflake 스키마", 
           "C) Snowflake 리전",
           "D) Snowflake 계정",
           "E) Snowflake 테이블"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...":
        ''
    else:
        answer = '27e8c3433312fce20b97b074e89590bf'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "아래 설명 중 Snowflake의 External Stage를 설명하는 것은 무엇입니까?"
options = ["아래 선택을 고르세요...",
           "A) 스테이지는 테이블의 각 새로운 로우에 대해 고유한 ID를 생성하는 데 사용할 수 있는 카운터로 사용할 수 있습니다. 시작 값과 증가분을 할당합니다", 
           "B) 스테이지는 데이터베이스 테이블을 그룹화하는 데 사용됩니다. 이를 위해 GARDEN_PLANTS 데이터베이스에 세 개의 스테이지를 생성했습니다. 그 중 하나는 VEGGIES라는 이름이 지정되었습니다", 
           "C) 스테이지는 Snowflake와 클라우드 폴더 간에 \"창\"을 제공할 수 있습니다"]                

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...":
        ''
    else:
        answer = 'e73deae50a17c8b5a30c514c9ef59a47'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "다음 중 Snowflake Cortex LLM Function이 아닌 것은 무엇인가요?"
options = ["아래 선택을 고르세요...",
           "A) TRANSLATE", 
           "B) SUMMARIZE", 
           "C) PARSE_DOCUMENT", 
           "D) COUNT_TOKENS", 
           "E) CONTENT", 
           "F) COMPLETE"]                

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...":
        ''
    else:
        answer = 'b71018c50689d88fc0f7aac8d3cbad47'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

## 축하합니다. :tada: :confetti_ball:

이 과정을 모두 마치셨습니다. 수고하셨습니다.
